# 6주차 과제: LangGraph 에이전트 아키텍처 I - 계획-실행 반복 + 툴 우선/복수 툴 호출

## 학습 목표
- 에이전트 루프(Plan→Act→Observe 반복)와 Tool Calling 구현
- LangGraph 기반 "툴-사용 에이전트" 구현
- 사용자 요청을 분석해 필요 시 툴을 호출하고, 결과를 반영해 여러 번 반복한 뒤 최종 답변 생성

## Part 1: 환경 설정 및 필요 라이브러리 설치

In [19]:
# 필요한 패키지 설치 (필요시 실행)
# !pip install -q langchain langchain-openai langgraph langchain-community python-dotenv
# !pip install -q tavily-python openweathermapy wikipedia-api numexpr

In [20]:
import os
import warnings
from dotenv import load_dotenv
from typing import Dict, List, TypedDict, Annotated, Optional, Union, Literal, Any
from operator import add
import json
from datetime import datetime
import asyncio

warnings.filterwarnings("ignore")

# .env 파일에서 환경 변수 로드
load_dotenv()

# API 키 확인
if os.getenv("OPENAI_API_KEY"):
    print(" * OpenAI API 키 로드 완료!")
else:
    print(" * 경고: OPENAI_API_KEY가 .env 파일에 설정되지 않았습니다.")

# Tavily API 키 설정 (웹 검색용)
if os.getenv("TAVILY_API_KEY"):  # .env 파일에 업데이트
    print(" * Tavily API 로드 완료!")
else:
    os.environ["TAVILY_API_KEY"] = "tvly-dummy-key"  # 테스트용 무료 키 (제한적 사용)
    print(" * Tavily API 키가 없어 테스트 키 사용")

# OpenWeatherMap API 키 확인
if os.getenv("OPENWEATHER_API_KEY"):
    print(" * OpenWeatherMap API 키 로드 완료!")
else:
    print(" * OpenWeatherMap API 키가 없어 시뮬레이션 모드 사용")

print("환경 설정 완료")

 * OpenAI API 키 로드 완료!
 * Tavily API 로드 완료!
 * OpenWeatherMap API 키 로드 완료!
환경 설정 완료


## Part 2: 상태(State) 정의

In [21]:
from typing import List, TypedDict, Sequence
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
    ToolMessage,
)
from langgraph.graph.message import add_messages


class PlanStep(TypedDict):
    """실행 계획의 각 단계를 정의"""

    step_number: int
    description: str
    tool_name: Optional[str]
    tool_args: Optional[Dict]
    status: Literal["pending", "completed", "failed"]
    result: Optional[str]


class AgentState(TypedDict):
    """에이전트 상태 정의"""

    messages: Annotated[Sequence[BaseMessage], add_messages]
    current_plan: List[PlanStep]
    current_step: int
    iteration_count: int
    max_iterations: int
    tool_results: List[Dict]
    final_answer: Optional[str]
    requires_refinement: bool


print("상태 클래스 정의 완료")

상태 클래스 정의 완료


## Part 3: 도구(Tools) 구현

In [22]:
!pip install numexpr wikipedia-api


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from langchain_core.tools import Tool
from langchain.tools import tool
import requests
import numexpr as ne
import wikipediaapi
from typing import Optional


# 1. 웹 검색 도구 (Tavily 또는 대체 구현)
@tool
def web_search(query: str) -> str:
    """웹에서 정보를 검색합니다.

    Args:
        query: 검색할 쿼리 문자열

    Returns:
        검색 결과 요약
    """
    try:
        # Tavily API 사용 시도
        from tavily import TavilyClient

        tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        results = tavily_client.search(query=query, max_results=3)

        search_results = []
        for result in results.get("results", [])[:3]:
            search_results.append(
                f"- {result.get('title', '')}: {result.get('content', '')}"
            )

        return (
            "\n".join(search_results)
            if search_results
            else "검색 결과를 찾을 수 없습니다."
        )
    except:
        # Tavily API 실패 시 대체 구현
        # 실제로는 다른 검색 API 사용 가능
        return f"'{query}'에 대한 검색 결과 (시뮬레이션):\n- 관련 정보 1\n- 관련 정보 2\n- 관련 정보 3"


# 2. 계산기 도구
@tool
def calculator(expression: str) -> str:
    """수식을 계산합니다.

    Args:
        expression: 계산할 수식 (예: "2 + 2 * 3")

    Returns:
        계산 결과
    """
    try:
        # numexpr를 사용한 안전한 수식 계산
        result = ne.evaluate(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


# 3. 날씨 정보 도구
@tool
def get_weather(location: str) -> str:
    """특정 위치의 날씨 정보를 가져옵니다.

    Args:
        location: 도시 이름 (예: "Seoul", "New York")

    Returns:
        날씨 정보
    """
    try:
        # OpenWeatherMap API 사용 
        api_key = os.getenv("OPENWEATHER_API_KEY")
        if api_key:
            print(f"  * OpenWeatherMap API 사용 중...")
            url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={api_key}&units=metric&lang=kr"
            response = requests.get(url)
            if response.status_code == 200:
                print(f"  * {location} 실제 날씨 데이터 수신 성공")
                data = response.json()
                weather = data["weather"][0]["description"]
                temp = data["main"]["temp"]
                feels_like = data["main"]["feels_like"]
                humidity = data["main"]["humidity"]

                # 현재 날짜 추가
                from datetime import datetime
                today = datetime.now().strftime("%Y년 %m월 %d일")

                return f"오늘 날짜: {today}\n{location}의 현재 날씨:\n- 날씨: {weather}\n- 온도: {temp}°C (체감: {feels_like}°C)\n- 습도: {humidity}%"
            else:
                print(f"  * API 응답 오류: {response.status_code}")  # 추가
        else:
            print(f"  * OpenWeatherMap API 키 없음, 시뮬레이션 모드")  # 추가

        # API 키가 없거나 실패 시 시뮬레이션
        import random
        from datetime import datetime

        current_month = datetime.now().month
        # 겨울(12-2월)
        if current_month in [12, 1, 2]:
            temp = random.randint(-5, 10)
        # 봄/가을(3-5월, 9-11월)
        elif current_month in [3, 4, 5, 9, 10, 11]:
            temp = random.randint(10, 20)
        # 여름(6-8월)
        else:
            temp = random.randint(20, 35)

        weather_conditions = ["맑음", "구름 조금", "흐림", "비"]
        weather = random.choice(weather_conditions)
        today = datetime.now().strftime("%Y년 %m월 %d일")

        return f"오늘 날짜: {today}\n{location}의 현재 날씨 (시뮬레이션):\n- 날씨: {weather}\n- 온도: {temp}°C\n- 습도: {random.randint(40, 80)}%"
    except Exception as e:
        return f"날씨 정보를 가져올 수 없습니다: {str(e)}"


# 4. Wikipedia 검색 도구 (선택)
@tool
def wikipedia_search(topic: str) -> str:
    """Wikipedia에서 정보를 검색합니다.

    Args:
        topic: 검색할 주제

    Returns:
        Wikipedia 요약 정보
    """
    try:
        wiki = wikipediaapi.Wikipedia("ko")
        page = wiki.page(topic)

        if page.exists():
            # 요약 반환 (처음 500자)
            summary = (
                page.summary[:500] + "..." if len(page.summary) > 500 else page.summary
            )
            return f"{topic}에 대한 Wikipedia 정보:\n{summary}"
        else:
            # 영어로 재시도
            wiki_en = wikipediaapi.Wikipedia("en")
            page_en = wiki_en.page(topic)
            if page_en.exists():
                summary = (
                    page_en.summary[:500] + "..."
                    if len(page_en.summary) > 500
                    else page_en.summary
                )
                return f"{topic}에 대한 Wikipedia 정보:\n{summary}"
            else:
                return f"{topic}에 대한 Wikipedia 페이지를 찾을 수 없습니다."
    except Exception as e:
        return f"Wikipedia 검색 오류: {str(e)}"


# 5. Python REPL 도구 (선택)
@tool
def python_repl(code: str) -> str:
    """Python 코드를 실행합니다.

    Args:
        code: 실행할 Python 코드

    Returns:
        실행 결과
    """
    try:
        # 안전한 실행을 위한 제한된 환경
        exec_globals = {"__builtins__": {}}
        exec_locals = {}

        # 기본 함수들만 허용
        allowed_builtins = [
            "abs",
            "all",
            "any",
            "len",
            "max",
            "min",
            "sum",
            "range",
            "str",
            "int",
            "float",
            "list",
            "dict",
        ]
        for func in allowed_builtins:
            exec_globals[func] = eval(func)

        # 코드 실행
        exec(code, exec_globals, exec_locals)

        # 결과 반환
        if "result" in exec_locals:
            return f"실행 결과: {exec_locals['result']}"
        else:
            return "코드가 실행되었습니다. (결과를 보려면 'result' 변수에 저장하세요)"
    except Exception as e:
        return f"실행 오류: {str(e)}"


# 도구 목록 생성
tools = [web_search, calculator, get_weather, wikipedia_search, python_repl]

# 도구 이름과 설명을 매핑
tool_dict = {tool.name: tool for tool in tools}

print(f"도구 {len(tools)}개 구현 완료:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

도구 5개 구현 완료:
  - web_search: 웹에서 정보를 검색합니다.

    Args:
        query: 검색할 쿼리 문자열

    Returns:
        검색 결과 요약
  - calculator: 수식을 계산합니다.

    Args:
        expression: 계산할 수식 (예: "2 + 2 * 3")

    Returns:
        계산 결과
  - get_weather: 특정 위치의 날씨 정보를 가져옵니다.

    Args:
        location: 도시 이름 (예: "Seoul", "New York")

    Returns:
        날씨 정보
  - wikipedia_search: Wikipedia에서 정보를 검색합니다.

    Args:
        topic: 검색할 주제

    Returns:
        Wikipedia 요약 정보
  - python_repl: Python 코드를 실행합니다.

    Args:
        code: 실행할 Python 코드

    Returns:
        실행 결과


## Part 4: LLM 및 에이전트 노드 구현

In [24]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from typing import Any, Dict

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=2000)

print("LLM 초기화 완료")

LLM 초기화 완료


In [25]:
# 계획 수립 노드
from datetime import datetime


def plan_node(state: AgentState) -> AgentState:
    """사용자 요청을 분석하고 실행 계획을 수립"""
    print("* 계획 수립 중...")

    messages = state["messages"]
    user_query = messages[-1].content if messages else ""

    # 현재 날짜 정보 추가
    from datetime import datetime
    current_date = datetime.now()
    current_year = current_date.year

    # 사용 가능한 도구 정보 생성
    tools_info = "\n".join([f"- {tool.name}: {tool.description}" for tool in tools])

    planning_prompt = f"""당신은 사용자 요청을 분석하고 실행 계획을 수립하는 에이전트입니다.
    현재 날짜: {current_date.strftime('%Y년 %m월 %d일')}
    현재 연도: {current_year}년

    사용자 요청: {user_query}

    사용 가능한 도구:
    {tools_info}

    위 요청을 처리하기 위한 단계별 실행 계획을 JSON 형식으로 작성하세요.
    각 단계는 하나의 도구를 사용하거나 단순한 작업이어야 합니다.

    JSON 형식:
    {{
        "steps": [
            {{
                "step_number": 1,
                "description": "단계 설명",
                "tool_name": "사용할 도구 이름 (없으면 null)",
                "tool_args": {{"인자명": "값"}}
            }}
        ],
        "expected_outcome": "예상 결과"
    }}

    계획을 작성하세요:"""

    try:
        response = llm.invoke(planning_prompt)

        # JSON 파싱
        import re

        json_text = response.content
        # JSON 블록 추출
        json_match = re.search(r"\{.*\}", json_text, re.DOTALL)
        if json_match:
            json_text = json_match.group()

        plan_data = json.loads(json_text)

        # PlanStep 형식으로 변환
        plan_steps = []
        for step in plan_data.get("steps", []):
            plan_steps.append(
                PlanStep(
                    step_number=step["step_number"],
                    description=step["description"],
                    tool_name=step.get("tool_name"),
                    tool_args=step.get("tool_args", {}),
                    status="pending",
                    result=None,
                )
            )

        print(f"* 계획 수립 완료: {len(plan_steps)}개 단계")
        for step in plan_steps:
            print(
                f"  {step['step_number']}. {step['description']} [{step['tool_name']}]"
            )

        state["current_plan"] = plan_steps
        state["current_step"] = 0

    except Exception as e:
        print(f"* 계획 수립 실패: {str(e)}")
        # 기본 계획 설정
        state["current_plan"] = []
        state["current_step"] = 0

    return state


# 도구 실행 노드
def tool_call_node(state: AgentState) -> AgentState:
    """현재 단계의 도구를 실행"""
    print(" * 도구 실행 중...")

    current_plan = state.get("current_plan", [])
    current_step = state.get("current_step", 0)

    if current_step >= len(current_plan):
        print("모든 단계 완료")
        return state

    step = current_plan[current_step]
    print(f"실행 단계 {step['step_number']}: {step['description']}")

    if step["tool_name"] and step["tool_name"] in tool_dict:
        try:
            # 도구 실행
            tool = tool_dict[step["tool_name"]]

            # tool_args가 딕셔너리인 경우와 단일 값인 경우 처리
            if isinstance(step["tool_args"], dict):
                # 딕셔너리에서 첫 번째 값 추출 또는 전체 딕셔너리 사용
                if len(step["tool_args"]) == 1:
                    # 단일 인자인 경우
                    arg_value = list(step["tool_args"].values())[0]
                    result = tool.invoke(arg_value)
                else:
                    # 여러 인자인 경우
                    result = tool.invoke(step["tool_args"])
            else:
                result = tool.invoke(step["tool_args"])

            # 결과 저장
            step["result"] = str(result)
            step["status"] = "completed"

            # tool_results에 추가
            state["tool_results"].append(
                {
                    "tool": step["tool_name"],
                    "input": step["tool_args"],
                    "output": result,
                }
            )

            print(f"* 도구 실행 성공: {step['tool_name']}")
            print(
                f"   결과: {result[:100]}..."
                if len(str(result)) > 100
                else f"   결과: {result}"
            )

        except Exception as e:
            print(f"* 도구 실행 실패: {str(e)}")
            step["status"] = "failed"
            step["result"] = f"오류: {str(e)}"
    else:
        # 도구 없이 단순 작업
        step["status"] = "completed"
        step["result"] = "단순 작업 완료"

    # 현재 단계 증가
    state["current_step"] = current_step + 1

    return state


# 결과 검증 노드
def check_result_node(state: AgentState) -> AgentState:
    """실행 결과를 검증하고 다음 단계 결정"""
    print("* 결과 검증 중...")

    current_plan = state.get("current_plan", [])
    current_step = state.get("current_step", 0)

    # 모든 단계가 완료되었는지 확인
    all_completed = all(
        step["status"] in ["completed", "failed"] for step in current_plan
    )

    # 실패한 단계가 있는지 확인
    has_failures = any(step["status"] == "failed" for step in current_plan)

    if all_completed:
        if has_failures:
            print("* 일부 단계 실패, 재시도 필요")
            state["requires_refinement"] = True
        else:
            print("* 모든 단계 성공적으로 완료")
            state["requires_refinement"] = False
    else:
        print(f"* 진행 중: {current_step}/{len(current_plan)} 단계 완료")
        state["requires_refinement"] = False

    # 반복 횟수 증가
    state["iteration_count"] = state.get("iteration_count", 0) + 1

    return state


# 계획 수정 노드
def refine_node(state: AgentState) -> AgentState:
    """실패한 단계를 수정하거나 새로운 계획 수립"""
    print("* 계획 수정 중...")

    current_plan = state.get("current_plan", [])
    failed_steps = [step for step in current_plan if step["status"] == "failed"]

    if failed_steps:
        print(f"실패한 단계 {len(failed_steps)}개 재시도 준비")

        # 실패한 단계를 pending으로 리셋
        for step in failed_steps:
            step["status"] = "pending"
            step["result"] = None

        # current_step을 첫 번째 실패한 단계로 설정
        first_failed_index = next(
            (i for i, step in enumerate(current_plan) if step in failed_steps), 0
        )
        state["current_step"] = first_failed_index

    state["requires_refinement"] = False
    return state


# 최종 답변 생성 노드
def generate_answer_node(state: AgentState) -> AgentState:
    """모든 결과를 종합하여 최종 답변 생성"""
    print("* 최종 답변 생성 중...")

    messages = state["messages"]
    user_query = messages[-1].content if messages else ""
    current_plan = state.get("current_plan", [])
    tool_results = state.get("tool_results", [])

    # 실행 결과 요약
    results_summary = "\n".join(
        [
            f"- {step['description']}: {step.get('result', '결과 없음')[:200]}"
            for step in current_plan
            if step.get("status") == "completed"
        ]
    )

    answer_prompt = f"""사용자 질문: {user_query}

실행된 작업과 결과:
{results_summary}

위 정보를 바탕으로 사용자 질문에 대한 종합적이고 자연스러운 답변을 작성하세요.
답변:"""

    response = llm.invoke(answer_prompt)
    final_answer = response.content

    # 상태에 최종 답변 저장
    state["final_answer"] = final_answer

    # 메시지에 추가
    state["messages"].append(AIMessage(content=final_answer))

    print("* 최종 답변 생성 완료")

    return state


print("노드 함수 구현 완료")

노드 함수 구현 완료


## Part 5: Tool-First Call 및 Multiple Tool Calling 구현

In [26]:
# Tool-First Call을 위한 패턴 매칭
def detect_tool_first_patterns(query: str) -> Optional[str]:
    """쿼리에서 특정 패턴을 감지하여 우선 호출할 도구 결정"""
    query_lower = query.lower()

    # 날씨 관련 키워드
    weather_keywords = ["날씨", "weather", "기온", "온도", "비", "눈", "맑음", "흐림"]
    if any(keyword in query_lower for keyword in weather_keywords):
        return "get_weather"

    # 계산 관련 키워드
    calc_keywords = [
        "계산",
        "calculate",
        "더하기",
        "빼기",
        "곱하기",
        "나누기",
        "+",
        "-",
        "*",
        "/",
        "=",
    ]
    if any(keyword in query_lower for keyword in calc_keywords):
        return "calculator"

    # 검색 관련 키워드
    search_keywords = ["검색", "search", "찾아", "find", "알아봐", "정보"]
    if any(keyword in query_lower for keyword in search_keywords):
        return "web_search"

    return None


# 병렬 도구 실행을 위한 비동기 함수
async def execute_tools_parallel(tool_calls: List[Dict]) -> List[Dict]:
    """여러 도구를 병렬로 실행"""

    async def execute_single_tool(tool_call: Dict) -> Dict:
        tool_name = tool_call["tool_name"]
        tool_args = tool_call["tool_args"]

        if tool_name in tool_dict:
            try:
                tool = tool_dict[tool_name]
                # 동기 함수를 비동기로 실행
                loop = asyncio.get_event_loop()
                result = await loop.run_in_executor(None, tool.invoke, tool_args)
                return {
                    "tool": tool_name,
                    "input": tool_args,
                    "output": result,
                    "status": "success",
                }
            except Exception as e:
                return {
                    "tool": tool_name,
                    "input": tool_args,
                    "error": str(e),
                    "status": "failed",
                }
        return {"error": f"Tool {tool_name} not found", "status": "failed"}

    # 모든 도구를 병렬로 실행
    results = await asyncio.gather(*[execute_single_tool(call) for call in tool_calls])
    return results


# 향상된 계획 수립 노드 (Tool-First 지원)
def enhanced_plan_node(state: AgentState) -> AgentState:
    """Tool-First 패턴을 고려한 계획 수립"""
    print("* 향상된 계획 수립 중...")

    from datetime import datetime
    current_year = datetime.now().year

    messages = state["messages"]
    user_query = messages[-1].content if messages else ""

    # Tool-First 패턴 검사
    priority_tool = detect_tool_first_patterns(user_query)

    if priority_tool:
        print(f"* Tool-First 감지: {priority_tool}")

        # 우선 도구를 첫 번째 단계로 설정
        if priority_tool == "get_weather":
            # 도시 이름 추출 (간단한 패턴 매칭)
            import re

            cities = re.findall(
                r"\b(?:인천|Incheon|서울|Seoul|뉴욕|New York|도쿄|Tokyo|런던|London|파리|Paris|부산|Busan)\b",
                user_query,
                re.IGNORECASE,
            )

            if cities:
                # 여러 도시가 언급된 경우 병렬 처리 계획
                plan_steps = []
                for i, city in enumerate(cities, 1):
                    plan_steps.append(
                        PlanStep(
                            step_number=i,
                            description=f"{city}의 날씨 정보 조회",
                            tool_name="get_weather",
                            tool_args={"location": city},
                            status="pending",
                            result=None,
                        )
                    )

                if len(cities) > 1:
                    # 비교 단계 추가
                    plan_steps.append(
                        PlanStep(
                            step_number=len(cities) + 1,
                            description="날씨 정보 비교 분석",
                            tool_name=None,
                            tool_args={},
                            status="pending",
                            result=None,
                        )
                    )

                state["current_plan"] = plan_steps
                state["current_step"] = 0
                return state

    # Tool-First가 아닌 경우 일반 계획 수립
    return plan_node(state)


print("Tool-First 및 병렬 실행 기능 구현 완료")

Tool-First 및 병렬 실행 기능 구현 완료


## Part 6: LangGraph 그래프 구성

In [27]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


# 라우팅 함수들
def should_continue(state: AgentState) -> str:
    """계속 진행할지 결정"""
    current_plan = state.get("current_plan", [])
    current_step = state.get("current_step", 0)
    iteration_count = state.get("iteration_count", 0)
    max_iterations = state.get("max_iterations", 10)

    # 최대 반복 횟수 초과
    if iteration_count >= max_iterations:
        print(f"* 최대 반복 횟수({max_iterations}) 도달")
        return "generate_answer"

    # 계획이 없는 경우
    if not current_plan:
        return "plan"

    # 모든 단계 완료
    if current_step >= len(current_plan):
        all_completed = all(step["status"] == "completed" for step in current_plan)
        if all_completed:
            return "generate_answer"
        elif state.get("requires_refinement", False):
            return "refine"
        else:
            return "generate_answer"

    # 다음 단계 실행
    return "tool_call"


def after_check(state: AgentState) -> str:
    """검증 후 다음 단계 결정"""
    if state.get("requires_refinement", False):
        return "refine"

    current_plan = state.get("current_plan", [])
    current_step = state.get("current_step", 0)

    if current_step >= len(current_plan):
        return "generate_answer"
    else:
        return "tool_call"


# StateGraph 생성
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("plan", enhanced_plan_node)  # Tool-First 지원 버전 사용
workflow.add_node("tool_call", tool_call_node)
workflow.add_node("check_result", check_result_node)
workflow.add_node("refine", refine_node)
workflow.add_node("generate_answer", generate_answer_node)

# 시작점 설정
workflow.set_entry_point("plan")

# 엣지 정의
workflow.add_conditional_edges(
    "plan",
    should_continue,
    {"plan": "plan", "tool_call": "tool_call", "generate_answer": "generate_answer"},
)

workflow.add_edge("tool_call", "check_result")

workflow.add_conditional_edges(
    "check_result",
    after_check,
    {
        "refine": "refine",
        "tool_call": "tool_call",
        "generate_answer": "generate_answer",
    },
)

workflow.add_edge("refine", "tool_call")
workflow.add_edge("generate_answer", END)

# 메모리 설정
memory = MemorySaver()

# 그래프 컴파일
app = workflow.compile(checkpointer=memory)

print("그래프 구성 완료!")
print("노드:", list(workflow.nodes.keys()))
print("엣지:", list(workflow.edges))

그래프 구성 완료!
노드: ['plan', 'tool_call', 'check_result', 'refine', 'generate_answer']
엣지: [('tool_call', 'check_result'), ('generate_answer', '__end__'), ('__start__', 'plan'), ('refine', 'tool_call')]


## Part 7: 테스트 시나리오 실행

In [28]:
# 에이전트 실행 함수
def run_agent(query: str, thread_id: str = "test-session") -> str:
    """에이전트를 실행하고 결과 반환"""
    print(f"\n{'='*60}")
    print(f"* 에이전트 실행: {query}")
    print(f"{'='*60}\n")

    # 초기 상태 설정
    initial_state = AgentState(
        messages=[HumanMessage(content=query)],
        current_plan=[],
        current_step=0,
        iteration_count=0,
        max_iterations=10,
        tool_results=[],
        final_answer=None,
        requires_refinement=False,
    )

    try:
        # 그래프 실행
        result = app.invoke(
            initial_state, config={"configurable": {"thread_id": thread_id}}
        )

        # 최종 답변 반환
        final_answer = result.get("final_answer", "답변 생성 실패")

        print(f"\n{'='*60}")
        print("* 최종 답변:")
        print(f"{'='*60}")
        print(final_answer)
        print(f"{'='*60}\n")

        return final_answer

    except Exception as e:
        error_msg = f"에이전트 실행 오류: {str(e)}"
        print(f"* {error_msg}")
        return error_msg

In [29]:
# 시나리오 1: 단일 도구 사용
print("\n" + "=" * 80)
print("시나리오 1: 단일 도구 사용")
print("=" * 80)

result1 = run_agent("오늘 인천 날씨 알려줘")


시나리오 1: 단일 도구 사용

* 에이전트 실행: 오늘 인천 날씨 알려줘

* 향상된 계획 수립 중...
* Tool-First 감지: get_weather
 * 도구 실행 중...
실행 단계 1: 인천의 날씨 정보 조회
  * OpenWeatherMap API 사용 중...
  * API 응답 오류: 401
* 도구 실행 성공: get_weather
   결과: 오늘 날짜: 2025년 12월 22일
인천의 현재 날씨 (시뮬레이션):
- 날씨: 맑음
- 온도: -1°C
- 습도: 78%
* 결과 검증 중...
* 모든 단계 성공적으로 완료
* 최종 답변 생성 중...
* 최종 답변 생성 완료

* 최종 답변:
오늘 인천의 날씨는 맑고 기온은 -1°C입니다. 습도는 78%로 다소 습한 편입니다. 외출 시 따뜻한 옷을 챙기시는 것이 좋겠습니다!



In [30]:
# 시나리오 2: 복수 도구 순차 실행
print("\n" + "=" * 80)
print("시나리오 2: 복수 도구 순차 실행")
print("=" * 80)

result2 = run_agent("파이썬 창시자가 태어난 해와 올해의 차이를 계산해줘")


시나리오 2: 복수 도구 순차 실행

* 에이전트 실행: 파이썬 창시자가 태어난 해와 올해의 차이를 계산해줘

* 향상된 계획 수립 중...
* Tool-First 감지: calculator
* 계획 수립 중...
* 계획 수립 완료: 3개 단계
  1. 파이썬 창시자인 귀도 반 로섬의 생년을 찾기 위해 웹 검색을 수행합니다. [web_search]
  2. 웹 검색 결과에서 귀도 반 로섬의 생년을 추출합니다. [None]
  3. 현재 연도와 귀도 반 로섬의 생년 사이의 차이를 계산합니다. [calculator]
 * 도구 실행 중...
실행 단계 1: 파이썬 창시자인 귀도 반 로섬의 생년을 찾기 위해 웹 검색을 수행합니다.
* 도구 실행 성공: web_search
   결과: 'Guido van Rossum birth year'에 대한 검색 결과 (시뮬레이션):
- 관련 정보 1
- 관련 정보 2
- 관련 정보 3
* 결과 검증 중...
* 진행 중: 1/3 단계 완료
 * 도구 실행 중...
실행 단계 2: 웹 검색 결과에서 귀도 반 로섬의 생년을 추출합니다.
* 결과 검증 중...
* 진행 중: 2/3 단계 완료
 * 도구 실행 중...
실행 단계 3: 현재 연도와 귀도 반 로섬의 생년 사이의 차이를 계산합니다.
* 도구 실행 성공: calculator
   결과: 계산 오류: Expression 2025 - [귀도 반 로섬의 생년] has forbidden control characters.
* 결과 검증 중...
* 모든 단계 성공적으로 완료
* 최종 답변 생성 중...
* 최종 답변 생성 완료

* 최종 답변:
귀도 반 로섬(Guido van Rossum)은 1956년에 태어났습니다. 현재 연도는 2023년이므로, 2023년과 그의 생년인 1956년 사이의 차이는 2023 - 1956 = 67년입니다. 따라서 귀도 반 로섬은 현재 67세입니다.



In [31]:
# 시나리오 3: 복수 도구 병렬 실행
print("\n" + "=" * 80)
print("시나리오 3: 복수 도구 병렬 실행")
print("=" * 80)

result3 = run_agent("인천과 서울의 현재 날씨를 비교해줘")


시나리오 3: 복수 도구 병렬 실행

* 에이전트 실행: 인천과 서울의 현재 날씨를 비교해줘

* 향상된 계획 수립 중...
* Tool-First 감지: get_weather
* 계획 수립 중...
* 계획 수립 완료: 3개 단계
  1. 인천의 현재 날씨 정보를 가져옵니다. [get_weather]
  2. 서울의 현재 날씨 정보를 가져옵니다. [get_weather]
  3. 인천과 서울의 날씨 정보를 비교합니다. [null]
 * 도구 실행 중...
실행 단계 1: 인천의 현재 날씨 정보를 가져옵니다.
  * OpenWeatherMap API 사용 중...
  * API 응답 오류: 401
* 도구 실행 성공: get_weather
   결과: 오늘 날짜: 2025년 12월 22일
Incheon의 현재 날씨 (시뮬레이션):
- 날씨: 맑음
- 온도: 7°C
- 습도: 69%
* 결과 검증 중...
* 진행 중: 1/3 단계 완료
 * 도구 실행 중...
실행 단계 2: 서울의 현재 날씨 정보를 가져옵니다.
  * OpenWeatherMap API 사용 중...
  * API 응답 오류: 401
* 도구 실행 성공: get_weather
   결과: 오늘 날짜: 2025년 12월 22일
Seoul의 현재 날씨 (시뮬레이션):
- 날씨: 구름 조금
- 온도: 2°C
- 습도: 46%
* 결과 검증 중...
* 진행 중: 2/3 단계 완료
 * 도구 실행 중...
실행 단계 3: 인천과 서울의 날씨 정보를 비교합니다.
* 결과 검증 중...
* 모든 단계 성공적으로 완료
* 최종 답변 생성 중...
* 최종 답변 생성 완료

* 최종 답변:
현재 인천과 서울의 날씨를 비교해보면 다음과 같습니다.

**인천**:
- 날씨: 맑음
- 온도: 7°C
- 습도: 69%

**서울**:
- 날씨: 구름 조금
- 온도: 2°C
- 습도: 46%

인천은 맑은 날씨에 온도가 7°C로 서울보다 따뜻하며, 습도는 69%로 상대적으로 높은 편입니다. 반면 서울은 구름이 조금

In [32]:
# 시나리오 4: 계획 수정 시나리오
print("\n" + "=" * 80)
print("시나리오 4: 계획 수정 시나리오")
print("=" * 80)

result4 = run_agent(
    "최근 AI 뉴스 3개를 찾아서 요약하고, 가장 중요한 것의 상세 정보를 알려줘"
)


시나리오 4: 계획 수정 시나리오

* 에이전트 실행: 최근 AI 뉴스 3개를 찾아서 요약하고, 가장 중요한 것의 상세 정보를 알려줘

* 향상된 계획 수립 중...
* Tool-First 감지: web_search
* 계획 수립 중...
* 계획 수립 완료: 3개 단계
  1. 최근 AI 뉴스 3개를 검색합니다. [web_search]
  2. 검색 결과에서 뉴스 3개의 요약을 작성합니다. [None]
  3. 가장 중요한 뉴스의 상세 정보를 추가로 검색합니다. [web_search]
 * 도구 실행 중...
실행 단계 1: 최근 AI 뉴스 3개를 검색합니다.
* 도구 실행 성공: web_search
   결과: '최근 AI 뉴스'에 대한 검색 결과 (시뮬레이션):
- 관련 정보 1
- 관련 정보 2
- 관련 정보 3
* 결과 검증 중...
* 진행 중: 1/3 단계 완료
 * 도구 실행 중...
실행 단계 2: 검색 결과에서 뉴스 3개의 요약을 작성합니다.
* 결과 검증 중...
* 진행 중: 2/3 단계 완료
 * 도구 실행 중...
실행 단계 3: 가장 중요한 뉴스의 상세 정보를 추가로 검색합니다.
* 도구 실행 성공: web_search
   결과: '가장 중요한 AI 뉴스의 상세 정보'에 대한 검색 결과 (시뮬레이션):
- 관련 정보 1
- 관련 정보 2
- 관련 정보 3
* 결과 검증 중...
* 모든 단계 성공적으로 완료
* 최종 답변 생성 중...
* 최종 답변 생성 완료

* 최종 답변:
최근 AI 관련 뉴스 3개를 요약해 드리겠습니다.

1. **AI의 윤리적 사용에 대한 국제 회의**: 여러 국가의 정부와 기업들이 모여 AI의 윤리적 사용과 규제 방안에 대해 논의했습니다. 회의에서는 AI 기술이 사회에 미치는 영향과 이를 관리하기 위한 국제적인 협력의 필요성이 강조되었습니다.

2. **AI 기반 의료 진단 시스템의 발전**: 한 연구팀이 AI를 이용한 새로운 의료 진단 시스템을 개발했습니다. 이 시스템은 기존의 진단 방법보다 더 빠르고

## Part 8: 고급 기능 - 도구 간 의존성 관리

In [33]:
# 도구 의존성 관리
class ToolDependencyManager:
    """도구 간 의존성을 관리하는 클래스"""

    def __init__(self):
        self.dependencies = {
            # 예: "tool_name": ["required_tool1", "required_tool2"]
        }

    def check_dependencies(self, plan_steps: List[PlanStep]) -> bool:
        """계획의 의존성이 올바른지 확인"""
        completed_tools = set()

        for step in plan_steps:
            if step["tool_name"]:
                # 의존성 확인
                required_tools = self.dependencies.get(step["tool_name"], [])
                for req_tool in required_tools:
                    if req_tool not in completed_tools:
                        print(
                            f"⚠️ 의존성 오류: {step['tool_name']}는 {req_tool}를 먼저 실행해야 합니다."
                        )
                        return False

                completed_tools.add(step["tool_name"])

        return True

    def optimize_plan(self, plan_steps: List[PlanStep]) -> List[PlanStep]:
        """의존성을 고려하여 계획을 최적화"""
        # 병렬 실행 가능한 단계 식별
        parallel_groups = []
        current_group = []

        for step in plan_steps:
            # 의존성이 없는 도구들은 같은 그룹에 배치
            if step["tool_name"] and not self.dependencies.get(step["tool_name"]):
                current_group.append(step)
            else:
                if current_group:
                    parallel_groups.append(current_group)
                    current_group = []
                parallel_groups.append([step])

        if current_group:
            parallel_groups.append(current_group)

        # 그룹별로 재구성
        optimized_plan = []
        step_number = 1

        for group in parallel_groups:
            for step in group:
                step["step_number"] = step_number
                step["parallel_group"] = step_number if len(group) > 1 else None
                optimized_plan.append(step)
            step_number += 1

        return optimized_plan


# 의존성 매니저 인스턴스 생성
dependency_manager = ToolDependencyManager()

print("도구 의존성 관리 시스템 구현 완료")

도구 의존성 관리 시스템 구현 완료


## Part 9: 성능 모니터링 및 평가

In [34]:
import time
from dataclasses import dataclass
from typing import List, Dict


@dataclass
class ExecutionMetrics:
    """실행 메트릭을 저장하는 클래스"""

    query: str
    total_time: float
    tool_calls: int
    successful_calls: int
    failed_calls: int
    iterations: int
    plan_steps: int
    final_answer_length: int


class PerformanceMonitor:
    """성능을 모니터링하는 클래스"""

    def __init__(self):
        self.metrics: List[ExecutionMetrics] = []

    def monitor_execution(self, query: str) -> Dict:
        """에이전트 실행을 모니터링"""
        start_time = time.time()

        # 에이전트 실행
        thread_id = f"monitor-{int(time.time())}"

        initial_state = AgentState(
            messages=[HumanMessage(content=query)],
            current_plan=[],
            current_step=0,
            iteration_count=0,
            max_iterations=10,
            tool_results=[],
            final_answer=None,
            requires_refinement=False,
        )

        try:
            result = app.invoke(
                initial_state, config={"configurable": {"thread_id": thread_id}}
            )

            # 메트릭 수집
            elapsed_time = time.time() - start_time

            tool_results = result.get("tool_results", [])
            successful_calls = sum(
                1 for r in tool_results if r.get("status") != "failed"
            )
            failed_calls = len(tool_results) - successful_calls

            metrics = ExecutionMetrics(
                query=query,
                total_time=elapsed_time,
                tool_calls=len(tool_results),
                successful_calls=successful_calls,
                failed_calls=failed_calls,
                iterations=result.get("iteration_count", 0),
                plan_steps=len(result.get("current_plan", [])),
                final_answer_length=len(result.get("final_answer", "")),
            )

            self.metrics.append(metrics)

            return {
                "success": True,
                "metrics": metrics,
                "answer": result.get("final_answer", ""),
            }

        except Exception as e:
            elapsed_time = time.time() - start_time

            metrics = ExecutionMetrics(
                query=query,
                total_time=elapsed_time,
                tool_calls=0,
                successful_calls=0,
                failed_calls=0,
                iterations=0,
                plan_steps=0,
                final_answer_length=0,
            )

            self.metrics.append(metrics)

            return {"success": False, "error": str(e), "metrics": metrics}

    def generate_report(self) -> str:
        """성능 보고서 생성"""
        if not self.metrics:
            return "측정된 메트릭이 없습니다."

        avg_time = sum(m.total_time for m in self.metrics) / len(self.metrics)
        avg_tools = sum(m.tool_calls for m in self.metrics) / len(self.metrics)
        success_rate = (
            sum(m.successful_calls for m in self.metrics)
            / max(sum(m.tool_calls for m in self.metrics), 1)
            * 100
        )

        report = f"""
📊 성능 보고서
{'='*50}
총 실행 횟수: {len(self.metrics)}
평균 실행 시간: {avg_time:.2f}초
평균 도구 호출 횟수: {avg_tools:.1f}
도구 호출 성공률: {success_rate:.1f}%

세부 메트릭:
"""

        for i, metric in enumerate(self.metrics, 1):
            report += f"""
{i}. {metric.query[:50]}...
   - 실행 시간: {metric.total_time:.2f}초
   - 도구 호출: {metric.tool_calls} (성공: {metric.successful_calls}, 실패: {metric.failed_calls})
   - 반복 횟수: {metric.iterations}
   - 계획 단계: {metric.plan_steps}
   - 답변 길이: {metric.final_answer_length}자
"""

        return report


# 성능 모니터 인스턴스 생성
monitor = PerformanceMonitor()

print("성능 모니터링 시스템 구현 완료")

성능 모니터링 시스템 구현 완료


In [35]:
# 평가 테스트 실행
test_queries = [
    "서울 날씨 알려줘",
    "100 + 200 * 3 계산해줘",
    "파이썬이 뭔지 설명해줘",
    "서울과 도쿄 날씨 비교해줘",
]

print("평가 테스트 시작...\n")

for query in test_queries:
    print(f"테스트: {query}")
    result = monitor.monitor_execution(query)

    if result["success"]:
        print(f"✅ 성공 - 실행시간: {result['metrics'].total_time:.2f}초")
    else:
        print(f"❌ 실패 - 오류: {result.get('error', 'Unknown')}")
    print()

# 성능 보고서 출력
print(monitor.generate_report())

평가 테스트 시작...

테스트: 서울 날씨 알려줘
* 향상된 계획 수립 중...
* Tool-First 감지: get_weather
 * 도구 실행 중...
실행 단계 1: 서울의 날씨 정보 조회
  * OpenWeatherMap API 사용 중...
  * API 응답 오류: 401
* 도구 실행 성공: get_weather
   결과: 오늘 날짜: 2025년 12월 22일
서울의 현재 날씨 (시뮬레이션):
- 날씨: 흐림
- 온도: 5°C
- 습도: 68%
* 결과 검증 중...
* 모든 단계 성공적으로 완료
* 최종 답변 생성 중...
* 최종 답변 생성 완료
✅ 성공 - 실행시간: 1.48초

테스트: 100 + 200 * 3 계산해줘
* 향상된 계획 수립 중...
* Tool-First 감지: calculator
* 계획 수립 중...
* 계획 수립 완료: 1개 단계
  1. 주어진 수식을 계산하기 위해 계산기를 사용합니다. [calculator]
 * 도구 실행 중...
실행 단계 1: 주어진 수식을 계산하기 위해 계산기를 사용합니다.
* 도구 실행 성공: calculator
   결과: 100 + 200 * 3 = 700
* 결과 검증 중...
* 모든 단계 성공적으로 완료
* 최종 답변 생성 중...
* 최종 답변 생성 완료
✅ 성공 - 실행시간: 4.65초

테스트: 파이썬이 뭔지 설명해줘
* 향상된 계획 수립 중...
* 계획 수립 중...
* 계획 수립 완료: 1개 단계
  1. Python에 대한 정보를 Wikipedia에서 검색합니다. [wikipedia_search]
 * 도구 실행 중...
실행 단계 1: Python에 대한 정보를 Wikipedia에서 검색합니다.
* 도구 실행 성공: wikipedia_search
   결과: Wikipedia 검색 오류: Please, be nice to Wikipedia and specify user agent - https://meta.wikimedia.org/wi...
* 결과 검증 중..

## Part 10: 대화형 인터페이스

In [36]:
# 대화형 인터페이스
def interactive_chat():
    """대화형 챗봇 인터페이스"""
    print("\n" + "=" * 60)
    print("🤖 LangGraph 에이전트 챗봇")
    print("도구를 활용한 지능형 에이전트입니다.")
    print("종료하려면 'quit', 'exit' 또는 '종료'를 입력하세요.")
    print("=" * 60 + "\n")

    thread_id = f"chat-{int(time.time())}"

    while True:
        user_input = input("\n👤 You: ").strip()

        if user_input.lower() in ["quit", "exit", "종료", "q"]:
            print("\n👋 대화를 종료합니다. 감사합니다!")
            break

        if not user_input:
            print("💡 질문을 입력해주세요.")
            continue

        # 에이전트 실행
        print("\n🤖 Agent: ", end="")
        result = run_agent(user_input, thread_id)


# 대화형 인터페이스 실행 (주석 처리 - 필요시 실행)
# interactive_chat()